# Musanga V2 — Notebook 03 | UAV illumination harmonization

**Objective:** Quantify whether conservative global and smooth local corrections reduce recorded illumination variation, without destroying crown-level RGB signals.

**Scope:** Five native-resolution UAV images. All sources are **read-only**. The notebook is CPU-only and samples a modest number of image patches rather than reading or rewriting full orthomosaics. All output products go exclusively to `MyDrive/Colab Notebooks/Musanga/KVO_work/V2/03_illumination_harmonization/`.

**Three comparable treatments:** **E0** original native RGB; **E1** bounded global luminance gain, reference built only from four development sites; **E2** E1 plus large-scale valid-mask-aware Gaussian illumination-field correction (scale specified in metres). These are *candidate transformations*, not a predetermined improvement.

**Scientific cautions:** RGB digital numbers are not calibrated incident illumination, species and background composition differ between sites, and patches were brightness-stratified rather than sampled proportionally to image area. Visualized corrections may change true canopy colour; quantify chromaticity and texture. **For four-site LOSO model comparisons, all reference parameters must be learned separately from each training fold's three forests.** Luki's 364 crowns are reserved for external validation, not for setting corrections.

The numbered sections are independently inspectable after Setup.

## 00 — Setup and immutable UAV data sources

CPU-only. Read original native-resolution RGB mosaics; establish the V2-only destination and fixed experimental settings.

In [ ]:
%pip -q install numpy pandas rasterio scipy matplotlib geopandas pyogrio
from pathlib import Path
import numpy as np,pandas as pd,rasterio,matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter,sobel
from rasterio.windows import Window
from IPython.display import display
from google.colab import drive
import json,hashlib,math,warnings
MY=Path("/content/drive/MyDrive")
if not MY.exists():drive.mount("/content/drive")
ROOT=MY/"Colab Notebooks"/"Musanga"
GEO=ROOT/"data"/"Geo_Ref"
V2=ROOT/"KVO_work"/"V2"
OUT=V2/"03_illumination_harmonization"
OUT.mkdir(parents=True,exist_ok=True)
SITES={
 "MOS_1_2":"MOS_1_2_kvo.tif",
 "Yoko":"Yoko_kvo_georef.tif",
 "Yangambi":"Yangambi_georef.tif",
 "Baego":"Baego_georef.tif",
 "Luki":"Luki.tif"}
TRAIN_SITES=[s for s in SITES if s!="Luki"]
SEED=20261013
CANDIDATES_PER_SITE=100
PATCH=512
N_PER_STRATUM=4 # 4 dark, 4 middle and 4 bright per forest
MIN_VALID=.80
# Conservative, predeclared methods: no evaluation labels or performance used for tuning.
GLOBAL_GAIN_BOUNDS=(.85,1.15)
LOCAL_GAIN_BOUNDS=(.75,1.25)
ILLUM_SIGMA_METRES=8.0
print("Run mode: READ ONLY inputs, CPU only. Output:",OUT)
for site,name in SITES.items():
    assert (GEO/name).exists(),f"Missing unchanged native UAV image: {GEO/name}"
assert OUT.is_relative_to(V2),"All V2 output MUST remain beneath KVO_work/V2"


## 01 — Stratified UAV RGB patch sampling

Select spatially distributed valid 512×512 RGB crops, then choose dark, intermediate and bright patches **within each site** only to cover the site's dynamic range. Brightness measurements always use a common 0–1 index; sampling strata do not define the analysis scale.

In [ ]:
def valid_rgb03(rgb):
    return ~((rgb==0).all(axis=2)|(rgb==255).all(axis=2)|
             ((rgb[:,:,0]==0)&(rgb[:,:,1]==0)&(rgb[:,:,2]==255)))
def luminosity03(x):
    return (.2126*x[:,:,0]+.7152*x[:,:,1]+.0722*x[:,:,2])
def candidates03(height,width,n,patch,seed):
    rng=np.random.default_rng(seed)
    gy=max(2,int(np.ceil(np.sqrt(n*height/max(width,1)))))
    gx=max(2,int(np.ceil(n/gy)))
    for iy in range(gy):
        for ix in range(gx):
            row0=int(iy*max(0,height-patch)/gy)
            row1=int((iy+1)*max(0,height-patch)/gy)+1
            col0=int(ix*max(0,width-patch)/gx)
            col1=int((ix+1)*max(0,width-patch)/gx)+1
            r=int(rng.integers(row0,max(row0+1,row1)))
            c=int(rng.integers(col0,max(col0+1,col1)))
            yield max(0,min(height-patch,r)),max(0,min(width-patch,c))
RECORDS03=[];PATCHES03={};PIXEL_SIZE03={}
for si,(site,name) in enumerate(SITES.items()):
    available=[]
    with rasterio.open(GEO/name) as ds:
        assert ds.count>=3 and all(t=="uint8" for t in ds.dtypes[:3]),(site,ds.dtypes)
        assert ds.crs is not None and ds.crs.is_projected
        assert ds.width>=PATCH and ds.height>=PATCH
        gsd_x,gsd_y=abs(ds.transform.a),abs(ds.transform.e)
        gsd=float(np.sqrt(gsd_x*gsd_y))
        PIXEL_SIZE03[site]=gsd
        for k,(row,col) in enumerate(candidates03(ds.height,ds.width,CANDIDATES_PER_SITE,PATCH,SEED+si)):
            im=np.moveaxis(ds.read([1,2,3],window=Window(col,row,PATCH,PATCH)),0,-1)
            mask=valid_rgb03(im)
            if mask.mean()<MIN_VALID:continue
            x=im.astype(np.float32)/255.0
            median=float(np.median(luminosity03(x)[mask]))
            available.append(dict(row=row,col=col,median=median,
                                  valid_fraction=float(mask.mean())))
        if len(available)<3:
            raise RuntimeError(f"{site}: only {len(available)} valid candidates; cannot compare harmonization.")
        order=np.argsort([z["median"] for z in available])
        # disjoint brightness ranks for targeted but lightweight patch review
        strata={
            "dark":order[:min(N_PER_STRATUM,len(order)//3)],
            "middle":order[max(0,len(order)//2-N_PER_STRATUM//2):max(0,len(order)//2-N_PER_STRATUM//2)+N_PER_STRATUM],
            "bright":order[max(0,len(order)-N_PER_STRATUM):]}
        seen=set()
        for stratum,ii in strata.items():
            for p in ii:
                z=available[int(p)]
                key=(z["row"],z["col"])
                if key in seen:continue
                seen.add(key)
                im=np.moveaxis(ds.read([1,2,3],window=Window(z["col"],z["row"],PATCH,PATCH)),0,-1)
                valid=valid_rgb03(im)
                x=im.astype(np.float32)/255.
                pid=f"{site}_{z['row']}_{z['col']}"
                PATCHES03[pid]=(x,valid,site)
                RECORDS03.append(dict(patch_id=pid,site=site,row=z["row"],col=z["col"],
                    illumination_stratum=stratum,native_gsd_m=gsd,
                    median_luminance=z["median"],valid_fraction=z["valid_fraction"]))
    print(site,":",len(available),"valid candidates;",len([r for r in RECORDS03 if r["site"]==site]),
          "stratified patches; GSD",round(gsd,4),"m",flush=True)
PATCH_TABLE03=pd.DataFrame(RECORDS03)
PATCH_TABLE03.to_csv(OUT/"03_sampled_patch_registry.csv",index=False)
display(PATCH_TABLE03.groupby(["site","illumination_stratum"]).agg(
    patches=("patch_id","count"),median_brightness=("median_luminance","median"),
    gsd_m=("native_gsd_m","first")).round(3))


## 02 — Establish four-site RGB reference (without Luki)

Learn pooled reference medians from **MOS, Yoko, Yangambi and Baego only**, using balanced patch counts rather than pixels proportional to site area. Luki is held out. This estimates *recorded RGB brightness*, not physical incident irradiance.

In [ ]:
# Summaries use same per-patch valid RGB pixels. No site-specific normalization of test imagery.
per_site=[]
for site in SITES:
    patches=[(p,v) for p,v,s in PATCHES03.values() if s==site]
    luminance=np.concatenate([luminosity03(x)[mask] for x,mask in patches])
    # sampling: equal number of patches/site; patches are brightness stratified by design,
    # so these are NOT whole-orthomosaic distribution estimates
    rgb_medians=np.median(np.concatenate([x[m] for x,m in patches]),axis=0)
    per_site.append(dict(site=site,n_patches=len(patches),
        median_r=float(rgb_medians[0]),median_g=float(rgb_medians[1]),
        median_b=float(rgb_medians[2]),
        median_luminance=float(np.median(luminance)),
        p10_luminance=float(np.quantile(luminance,.10)),
        p90_luminance=float(np.quantile(luminance,.90))))
SITE_STATS03=pd.DataFrame(per_site)
SITE_STATS03.to_csv(OUT/"03_sampled_site_RGB_statistics.csv",index=False)
display(SITE_STATS03.round(3))
REFERENCE_LUMINANCE03=float(SITE_STATS03.set_index("site").loc[TRAIN_SITES,"median_luminance"].median())
REFERENCE_RGB03=np.array([
    SITE_STATS03.set_index("site").loc[TRAIN_SITES,f"median_{band}"].median()
    for band in "rgb"],dtype=np.float32)
print("Training-only reference luminance",round(REFERENCE_LUMINANCE03,4))
print("Training-only reference RGB",REFERENCE_RGB03)
print("NOTE: diagnostics from deliberately stratified patches; not representative whole-site means.")


## 03 — Conservative harmonization operators

Compare original RGB (E0), clipped global luminance gain (E1), and global gain plus smooth local illumination correction (E2). All corrections are multiplicative to largely retain chromaticity. The broad local field is estimated with validity-weighted Gaussian smoothing and metre-based scale; gains are bounded to avoid overcompensation.

In [ ]:
def global_gain03(x,mask,reference):
    before=luminosity03(x)
    median=float(np.median(before[mask]))
    gain=float(np.clip(reference/max(median,1e-5),*GLOBAL_GAIN_BOUNDS))
    out=np.clip(x*gain,0,1).astype(np.float32)
    out[~mask]=x[~mask]
    return out,dict(global_gain=gain,median_before=median,median_after=float(np.median(luminosity03(out)[mask])))
def local_correct03(x,mask,gsd,sigma_m=ILLUM_SIGMA_METRES):
    lum=luminosity03(x)
    # Mask-aware, large-scale Gaussian illumination estimate.
    sigma_px=max(1.,float(sigma_m/gsd))
    weights=gaussian_filter(mask.astype(np.float32),sigma=sigma_px,mode="reflect")
    smooth=gaussian_filter(np.where(mask,lum,0),sigma=sigma_px,mode="reflect")/np.maximum(weights,.03)
    anchor=float(np.median(smooth[mask]))
    ratio=anchor/np.maximum(smooth,.025)
    gain=np.clip(ratio,*LOCAL_GAIN_BOUNDS).astype(np.float32)
    result=np.clip(x*gain[:,:,None],0,1).astype(np.float32)
    result[~mask]=x[~mask]
    return result,dict(local_gain_p05=float(np.quantile(gain[mask],.05)),
        local_gain_p50=float(np.median(gain[mask])),
        local_gain_p95=float(np.quantile(gain[mask],.95)),
        local_gain_clipped_pct=float(100*np.mean((ratio[mask]<LOCAL_GAIN_BOUNDS[0])|
                                                (ratio[mask]>LOCAL_GAIN_BOUNDS[1]))),
        sigma_m=sigma_m,sigma_px=sigma_px)
def transform03(x,mask,site,mode):
    if mode=="E0_native":return x.copy(),dict(global_gain=1.,sigma_m=0.)
    y,g=global_gain03(x,mask,REFERENCE_LUMINANCE03)
    if mode=="E1_global":return y,g
    if mode=="E2_global_local":
        z,local=local_correct03(y,mask,PIXEL_SIZE03[site])
        return z,{**g,**local}
    raise ValueError(mode)
METHODS03=["E0_native","E1_global","E2_global_local"]


## 04 — Objective QA: distribution, texture, clipping and colour stability

Quantify before/after brightness, local luminance variability, chromaticity, edge preservation, saturation, clipping and gain limits. These measurements reveal *how much* imagery changes but do not prove better species identification. Aggressive corrections will be rejected before model training.

In [ ]:
def qa03(original,corrected,valid):
    a=luminosity03(original)[valid];b=luminosity03(corrected)[valid]
    clipped=np.any((corrected[valid]>=.999)&(original[valid]<.999),axis=1)
    # Normalized colour coordinates; safe for low-brightness values.
    chrom_a=original[valid]/np.maximum(original[valid].sum(axis=1,keepdims=True),1e-5)
    chrom_b=corrected[valid]/np.maximum(corrected[valid].sum(axis=1,keepdims=True),1e-5)
    chrom_shift=np.mean(np.abs(chrom_a-chrom_b))
    # Neighbor differences, restricted to interior valid pixels.
    lum_a=luminosity03(original);lum_b=luminosity03(corrected)
    gx_a=lum_a[:,1:]-lum_a[:,:-1]
    gx_b=lum_b[:,1:]-lum_b[:,:-1]
    vx=valid[:,1:]&valid[:,:-1]
    texture_ratio=float(np.std(gx_b[vx])/max(np.std(gx_a[vx]),1e-6))
    texture_corr=float(np.corrcoef(gx_a[vx],gx_b[vx])[0,1]) if vx.sum()>10 and np.std(gx_a[vx])>1e-6 and np.std(gx_b[vx])>1e-6 else np.nan
    return dict(median_after=float(np.median(b)),median_before=float(np.median(a)),
        luminance_p10=float(np.quantile(b,.10)),luminance_p90=float(np.quantile(b,.90)),
        luminance_sd=float(b.std()),brightness_change=float(np.median(b)-np.median(a)),
        introduced_saturated_pixel_pct=float(100*clipped.mean()),
        mean_abs_chromaticity_shift=float(chrom_shift),
        local_edge_std_ratio=texture_ratio,local_edge_correlation=texture_corr)
RESULT03=[];PREVIEWS03={}
for idx,row in PATCH_TABLE03.iterrows():
    pid=row.patch_id
    original,mask,site=PATCHES03[pid]
    PREVIEWS03[pid]={}
    for method in METHODS03:
        transformed,params=transform03(original,mask,site,method)
        stats=qa03(original,transformed,mask)
        # Transformation metadata and QA both contain median_before/median_after.
        # Prefix operator parameters to retain both sets without duplicate keyword errors.
        prefixed_params={f"operator_{k}":v for k,v in params.items()}
        RESULT03.append(dict(patch_id=pid,site=site,stratum=row.illumination_stratum,
            method=method,valid_fraction=float(mask.mean()),**stats,**prefixed_params))
        PREVIEWS03[pid][method]=transformed
QA_TABLE03=pd.DataFrame(RESULT03)
QA_TABLE03.to_csv(OUT/"03_harmonization_patch_metrics.csv",index=False)
AGG03=QA_TABLE03.groupby(["site","method"],as_index=False).agg(
    n=("patch_id","count"),median_luminance=("median_after","median"),
    mean_clipped_pct=("introduced_saturated_pixel_pct","mean"),
    median_chromaticity_shift=("mean_abs_chromaticity_shift","median"),
    median_edge_ratio=("local_edge_std_ratio","median"),
    median_edge_corr=("local_edge_correlation","median"))
AGG03.to_csv(OUT/"03_harmonization_site_summary.csv",index=False)
display(AGG03.round(4))
print("NOTE: Numeric QA here tests image transformation only; crown accuracy is assessed later.")


## 05 — Before/after diagnostic galleries

Show dark, medium and bright sampled patches for each forest on **identical RGB display scales** and save one gallery per site. Avoid automatic display normalization, which would conceal brightness differences.

In [ ]:
for site in SITES:
    chosen=[]
    for st in ("dark","middle","bright"):
        sub=PATCH_TABLE03[(PATCH_TABLE03.site==site)&(PATCH_TABLE03.illumination_stratum==st)]
        if len(sub):chosen.append(sub.iloc[len(sub)//2].patch_id)
    fig,axs=plt.subplots(len(chosen),3,figsize=(11,3.4*len(chosen)),squeeze=False,
                         constrained_layout=True)
    for i,pid in enumerate(chosen):
        for j,method in enumerate(METHODS03):
            axs[i,j].imshow(PREVIEWS03[pid][method],vmin=0,vmax=1)
            axs[i,j].set_title(f"{site} | {PATCH_TABLE03.set_index('patch_id').loc[pid,'illumination_stratum']} | {method}")
            axs[i,j].axis("off")
    figure=OUT/f"03_{site}_before_after_gallery.png"
    fig.savefig(figure,dpi=140,bbox_inches="tight")
    display(fig);plt.close(fig)
    print("Saved gallery:",figure,flush=True)


## 06 — Export reproducible settings and next-stage contract

Store transformation parameters and patch IDs, not transformed full-sized orthomosaics. Subsequent training must learn correction/reference statistics from training folds only, never the held-out test forest. Treat target-site image-statistic adaptation as a separately labelled experiment.

In [ ]:
# Independent-source safeguards: do not pick E1/E2 on the basis of external Luki accuracy.
# In full LOSO, a different reference must be recalculated from each fold's 3 TRAINING sites;
# the global four-site reference here is only a visual diagnostic (not valid for held-out 4-site LOSO).
CONTRACT03=dict(
    notebook="V2/03 UAV Illumination Harmonization",
    output_root=str(OUT),
    authoritative_uav={s:str(GEO/name) for s,name in SITES.items()},
    methods=METHODS03,
    seed=SEED,patch_size_px=PATCH,per_stratum=N_PER_STRATUM,
    sample_strategy="stratified valid-image windows within sites (not full-mosaic representativeness)",
    reference_training_sites=TRAIN_SITES,
    reference_luminance=float(REFERENCE_LUMINANCE03),
    reference_rgb=[float(x) for x in REFERENCE_RGB03],
    formula_luminance="(0.2126*R + 0.7152*G + 0.0722*B)/255, original uint8 channel values",
    global_gain_bounds=list(GLOBAL_GAIN_BOUNDS),
    local_gain_bounds=list(LOCAL_GAIN_BOUNDS),
    smooth_illumination_scale_m=ILLUM_SIGMA_METRES,
    evaluation="NO MODEL ACCURACY ESTIMATED; transformations only; independent Luki retained",
    strict_loso="for each held-out forest learn all cross-site reference parameters from the other 3 forests only",
    luki_rule="excluded from training, cross-site reference fitting and hyperparameter selection",
    original_sources="never modified; corrected patches held in RAM and diagnostic PNG only")
(OUT/"03_harmonization_contract.json").write_text(json.dumps(CONTRACT03,indent=2))
print("V2 NOTEBOOK 03 COMPLETE:",OUT)
print("Next step: inspect galleries and QA table, then choose a SMALL set of training variants.")
print("Do not save or regenerate huge corrected native mosaics yet.")
